<a href="https://colab.research.google.com/github/josippreva/RVID-projekt/blob/main/RVID_projekt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
from pathlib import Path
from PIL import Image

BASE = Path("/content/drive/MyDrive/RVID Projekt")

splits = {
    "train": BASE / "train" / "_annotations.coco.json",
    "valid": BASE / "valid" / "_annotations.coco.json",
    "test":  BASE / "test"  / "_annotations.coco.json",
}

def coco_to_yolo_labels(coco_json: Path, images_dir: Path, out_labels_dir: Path):
    out_labels_dir.mkdir(parents=True, exist_ok=True)
    data = json.loads(coco_json.read_text())

    # classes
    cats = sorted(data.get("categories", []), key=lambda c: c["id"])
    cat_id_to_idx = {c["id"]: i for i, c in enumerate(cats)}
    class_names = [c["name"] for c in cats]

    # images
    img_map = {}
    for im in data.get("images", []):
        img_map[im["id"]] = (im["file_name"], im.get("width"), im.get("height"))

    # group anns
    ann_by_img = {}
    for ann in data.get("annotations", []):
        if ann.get("iscrowd", 0) == 1:
            continue
        ann_by_img.setdefault(ann["image_id"], []).append(ann)

    written, missing_imgs, total_boxes = 0, 0, 0

    for img_id, (fname, w, h) in img_map.items():
        src = images_dir / fname
        if not src.exists():
            src = images_dir / Path(fname).name
            if not src.exists():
                missing_imgs += 1
                continue

        if not w or not h:
            with Image.open(src) as im:
                w, h = im.size

        label_path = out_labels_dir / (Path(src).stem + ".txt")
        lines = []

        for ann in ann_by_img.get(img_id, []):
            cls = cat_id_to_idx.get(ann["category_id"], None)
            if cls is None:
                continue
            x, y, bw, bh = ann["bbox"]

            xc = (x + bw / 2) / w
            yc = (y + bh / 2) / h
            nw = bw / w
            nh = bh / h

            if nw <= 0 or nh <= 0:
                continue

            lines.append(f"{cls} {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}")
            total_boxes += 1

        label_path.write_text("\n".join(lines))
        written += 1

    return class_names, written, missing_imgs, total_boxes

all_classes = None
stats = {}

for split in ["train", "valid", "test"]:
    labels_dir = BASE / "labels" / split
    classes, written, missing, boxes = coco_to_yolo_labels(
        coco_json=splits[split],
        images_dir=BASE / split,
        out_labels_dir=labels_dir
    )
    stats[split] = {
        "images_labeled": written,
        "missing_images": missing,
        "boxes_total": boxes
    }
    if all_classes is None:
        all_classes = classes

print("✅ YOLO labele generirane (bez kopiranja slika).")
print("📊 Statistika:", stats)
print("🏷️ Klase:", all_classes)


✅ YOLO labele generirane (bez kopiranja slika).
📊 Statistika: {'train': {'images_labeled': 8691, 'missing_images': 0, 'boxes_total': 497856}, 'valid': {'images_labeled': 2483, 'missing_images': 0, 'boxes_total': 143316}, 'test': {'images_labeled': 1241, 'missing_images': 1, 'boxes_total': 70584}}
🏷️ Klase: ['spaces', 'space-empty', 'space-occupied']


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [ ]:
import os, shutil, glob
from pathlib import Path

BASE = Path("/content/drive/MyDrive/RVID Projekt")


DS = Path("/content/ds")
for p in [DS/"images/train", DS/"images/val", DS/"images/test",
          DS/"labels/train", DS/"labels/val", DS/"labels/test"]:
    p.mkdir(parents=True, exist_ok=True)

def link_images(src_dir: Path, dst_dir: Path):
    img_exts = {".jpg",".jpeg",".png",".bmp",".webp"}
    imgs = [p for p in src_dir.iterdir() if p.suffix.lower() in img_exts]
    for p in imgs:
        dst = dst_dir / p.name
        if dst.exists():
            continue
        os.symlink(str(p), str(dst))

link_images(BASE/"train", DS/"images/train")
link_images(BASE/"valid", DS/"images/val")
link_images(BASE/"test",  DS/"images/test")

def copy_labels(src_dir: Path, dst_dir: Path):
    dst_dir.mkdir(parents=True, exist_ok=True)
    for p in src_dir.glob("*.txt"):
        dst = dst_dir / p.name
        if not dst.exists():
            shutil.copy2(p, dst)

copy_labels(BASE/"labels/train", DS/"labels/train")
copy_labels(BASE/"labels/valid", DS/"labels/val")
copy_labels(BASE/"labels/test",  DS/"labels/test")

print("Train images:", len(list((DS/"images/train").glob("*"))))
print("Val images  :", len(list((DS/"images/val").glob("*"))))
print("Test images :", len(list((DS/"images/test").glob("*"))))

print("Train labels:", len(list((DS/"labels/train").glob("*.txt"))))
print("Val labels  :", len(list((DS/"labels/val").glob("*.txt"))))
print("Test labels :", len(list((DS/"labels/test").glob("*.txt"))))

Train images: 8694
Val images  : 2483
Test images : 1241
Train labels: 8691
Val labels  : 2483
Test labels : 1241


In [ ]:
from pathlib import Path

yaml_path = Path("/content/ds/data.yaml")
yaml_text = """path: /content/ds
train: images/train
val: images/val
test: images/test

names:
  0: spaces
  1: space-empty
  2: space-occupied
"""
yaml_path.write_text(yaml_text)
print("✅ data.yaml:", yaml_path)
print(yaml_text)

✅ data.yaml: /content/ds/data.yaml
path: /content/ds
train: images/train
val: images/val
test: images/test

names:
  0: spaces
  1: space-empty
  2: space-occupied



In [ ]:
!pip -q install ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 79.2 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="/content/ds/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    device=0
)

Ultralytics 8.4.12 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/ds/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79b16131ab40>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/train3/weights/best.pt")
model.val(data="/content/ds/data.yaml", split="test")

Ultralytics 8.4.12 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.4±0.2 ms, read: 0.2±0.1 MB/s, size: 70.9 KB)
val: Scanning /content/ds/labels/test... 1241 images, 26 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1241/1241 2.0it/s 10:27
val: New cache created: /content/ds/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 78/78 2.7it/s 28.4s
                   all       1241      70584      0.997      0.997      0.994      0.977
           space-empty       1040      36555      0.995      0.996      0.994      0.981
        space-occupied        990      34029      0.999      0.998      0.994      0.973
Speed: 1.7ms preprocess, 4.1ms inference, 0.0ms loss, 3.0ms postprocess per image
Results saved to /content/runs/detect/val


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79b1610c2630>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [ ]:
model.predict(source="/content/ds/images/test", conf=0.25, save=True)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/1241 /content/ds/images/test/2012-09-11_16_48_36_jpg.rf.4ecc8c87c61680ccc73edc218a2c8d7d.jpg: 640x640 25 space-emptys, 77 space-occupieds, 9.2ms
image 2/1241 /content/ds/images/test/2012-09-12_06_36_36_jpg.rf.08869047c7e9f62f5ce9334546b52958.jpg: 640x640 100 space-emptys, 7.3ms
image 3/1241 /content/ds/images/test/2012-09-12_08_15_53_jpg.rf.025635f821fe24c12ab497efbfd3d35c.jpg: 640x640 7 space-emptys, 93 space-occupieds, 7.3ms
image 4/1241 /conte

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'spaces', 1: 'space-empty', 2: 'space-occupied'}
 obb: None
 orig_img: array([[[104, 128, 164],
         [106, 130, 166],
         [106, 131, 165],
         ...,
         [ 29,  68,  66],
         [ 33,  72,  70],
         [ 35,  74,  72]],
 
        [[105, 129, 165],
         [106, 130, 166],
         [105, 129, 165],
         ...,
         [ 31,  70,  68],
         [ 35,  74,  72],
         [ 37,  76,  74]],
 
        [[103, 129, 166],
         [104, 130, 166],
         [103, 129, 165],
         ...,
         [ 35,  74,  72],
         [ 38,  77,  75],
         [ 40,  79,  77]],
 
        ...,
 
        [[ 88,  79,  82],
         [ 84,  75,  78],
         [ 77,  68,  71],
         ...,
         [100, 108, 137],
         [ 87,  99, 133],
         [ 93, 109, 145]],
 
        [[ 82,  75,  80],
         [ 81,  74,  79],
         [ 75,  68,

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/RVID Projekt/runs_train3/weights/best.pt")
model.predict(
    source="/content/drive/MyDrive/RVID Projekt/test/2013-03-11_18_15_14_jpg.rf.c2b33e31d571fefba72ee5cc037b5d67.jpg",
    conf=0.25,
    save=True
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

image 1/1 /content/drive/MyDrive/RVID Projekt/test/2013-03-11_18_15_14_jpg.rf.c2b33e31d571fefba72ee5cc037b5d67.jpg: 640x640 10 space-emptys, 30 space-occupieds, 7.3ms
Speed: 7.2ms preprocess, 7.3ms inference, 37.4ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/detect/predict


[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'spaces', 1: 'space-empty', 2: 'space-occupied'}
 obb: None
 orig_img: array([[[ 49,  68,  81],
         [ 46,  65,  78],
         [ 43,  62,  75],
         ...,
         [ 24,  51,  47],
         [ 21,  51,  46],
         [ 21,  53,  48]],
 
        [[ 44,  61,  74],
         [ 40,  59,  72],
         [ 38,  55,  68],
         ...,
         [ 22,  49,  45],
         [ 18,  48,  43],
         [ 18,  50,  45]],
 
        [[ 37,  53,  65],
         [ 34,  52,  63],
         [ 32,  48,  60],
         ...,
         [ 21,  48,  45],
         [ 18,  47,  44],
         [ 20,  49,  46]],
 
        ...,
 
        [[ 53,  60,  63],
         [ 53,  60,  63],
         [ 52,  59,  62],
         ...,
         [ 28,  57,  78],
         [ 47,  75, 105],
         [ 55,  84, 115]],
 
        [[ 53,  60,  63],
         [ 53,  60,  63],
         [ 52,  59,

In [ ]:
from pathlib import Path src = Path("/content/runs/detect/train3")
dst = Path("/content/drive/MyDrive/RVID Projekt/runs_train3")

if dst.exists():
   shutil.rmtree(dst)

shutil.copytree(src, dst)

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/RVID Projekt/runs_train3/weights/best.pt")
model.predict(
    source="/content/drive/MyDrive/RVID Projekt/slike parkinga s interneta/parking1.jpg",
    conf=0.05,
    save=True
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
